# Agricultural Yield Prediction — Week 1

Exploratory analysis, preprocessing, and baseline regression models for the [Kaggle Agricultural Yield dataset](https://www.kaggle.com/datasets/zoya77/agricultural-yield-prediction-dataset).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid", palette="muted")

# paths relative to this notebook's location — works from any cwd
NB_DIR  = Path(__file__).parent if "__file__" in dir() else Path().resolve()
ROOT    = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

## Load Data

In [ ]:
df = pd.read_csv("../data/raw/Agri_yield_prediction.csv")
print(f"{df.shape[0]:,} rows  ×  {df.shape[1]} columns")

In [ ]:
df.head()

In [ ]:
# check column names and types at a glance
df.dtypes

## Data Quality Check

In [ ]:
# missing values per column
df.isnull().sum()

In [ ]:
# any duplicate rows?
print("Duplicate rows:", df.duplicated().sum())

## EDA

In [ ]:
# overall descriptive stats
df.describe()

### Target — Yield

In [ ]:
df['Yield'].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["Yield"], bins=40, color="steelblue", edgecolor="white")
ax.set(title="Yield Distribution", xlabel="Yield (tons/ha)", ylabel="Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_01_yield_distribution.png", dpi=150)
plt.show()

In [ ]:
# boxplot to check for outliers
fig, ax = plt.subplots(figsize=(3, 5))
ax.boxplot(df["Yield"], patch_artist=True,
           boxprops=dict(facecolor="steelblue", alpha=0.6))
ax.set(title="Yield — Outlier Check", ylabel="Yield")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_02_yield_boxplot.png", dpi=150)
plt.show()

### Numerical Features

In [ ]:
num_cols = df.select_dtypes(include="number").columns.drop("Yield").tolist()
print(f"{len(num_cols)} numerical features")
print(num_cols)

In [ ]:
# spot-check a handful of agronomically meaningful features
spot = ["Temperature", "Humidity", "Rainfall", "GDD", "NDVI", "Chlorophyll"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, spot):
    ax.hist(df[col].dropna(), bins=30, color="coral", edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel(col)
plt.suptitle("Key Feature Distributions", y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_03_key_features.png", dpi=150)
plt.show()

In [ ]:
# plot the rest of the numerical features
remaining = [c for c in num_cols if c not in spot]

ncols = 4
nrows = -(-len(remaining) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
for ax, col in zip(axes.flat, remaining):
    ax.hist(df[col].dropna(), bins=30, color="mediumpurple", edgecolor="white")
    ax.set_title(col, fontsize=9)
for ax in axes.flat[len(remaining):]:
    ax.set_visible(False)
plt.suptitle("Remaining Numerical Features", y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_04_remaining_features.png", dpi=150)
plt.show()

### Categorical Features

In [ ]:
cat_cols = [c for c in df.select_dtypes(include="object").columns
            if "Date" not in c]

for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].value_counts().to_string())

In [ ]:
# all categories are well-balanced — no dominant class
ncols = 4
nrows = -(-len(cat_cols) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))
for ax, col in zip(axes.flat, cat_cols):
    vc = df[col].value_counts()
    ax.bar(vc.index, vc.values, color="teal", edgecolor="white")
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=30)
for ax in axes.flat[len(cat_cols):]:
    ax.set_visible(False)
plt.suptitle("Categorical Feature Distributions")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_05_categorical_distributions.png", dpi=150)
plt.show()

### Yield by Categorical Feature

Checking if any categorical grouping shows a meaningful difference in Yield.

In [ ]:
# boxplot of Yield by Crop_Type — is there a yield gap across crops?
fig, ax = plt.subplots(figsize=(8, 5))
order = df.groupby("Crop_Type")["Yield"].median().sort_values(ascending=False).index
sns.boxplot(data=df, x="Crop_Type", y="Yield", order=order,
            hue="Crop_Type", palette="Set2", legend=False, ax=ax)
ax.set(title="Yield by Crop Type", xlabel="Crop Type", ylabel="Yield (tons/ha)")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_06_yield_by_croptype.png", dpi=150)
plt.show()

In [ ]:
# same check for Season, Soil_Type, Region, Fertilizer_Type
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
group_cols = ["Season", "Soil_Type", "Region", "Fertilizer_Type"]

for ax, col in zip(axes.flat, group_cols):
    order = df.groupby(col)["Yield"].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y="Yield", order=order,
                hue=col, palette="Set3", legend=False, ax=ax)
    ax.set_title(f"Yield by {col}")
    ax.set_xlabel(col)
    ax.tick_params(axis="x", rotation=20)

plt.suptitle("Yield Distribution by Categorical Features", y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_07_yield_by_categorical.png", dpi=150)
plt.show()

In [ ]:
# quick mean ± std table — confirms whether the differences are real or noise
group_cols_all = ["Crop_Type", "Season", "Soil_Type", "Region",
                  "Fertilizer_Type", "Pesticide_Usage", "Growth_Stage"]

rows = []
for col in group_cols_all:
    grp = df.groupby(col)["Yield"].agg(["mean", "std", "count"])
    grp.columns = ["Mean Yield", "Std", "Count"]
    grp["Feature"] = col
    rows.append(grp.reset_index().rename(columns={col: "Group"}))

summary = pd.concat(rows, ignore_index=True)[["Feature", "Group", "Mean Yield", "Std", "Count"]]
print(summary.to_string(index=False))

## Correlation Analysis

In [ ]:
corr = df.select_dtypes(include='number').corr()

In [ ]:
# heatmap — too many features to annotate, so keeping it clean
fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(corr, cmap="coolwarm", center=0,
            linewidths=0.3, ax=ax, cbar_kws={"shrink": 0.7})
ax.set_title("Feature Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_08_correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
# correlations with Yield — sorted by absolute value
yield_corr = corr["Yield"].drop("Yield").abs().sort_values(ascending=False)
print("Top 15 features by |r| with Yield:")
print(yield_corr.head(15).round(4))

In [ ]:
# correlations are very weak (max |r| ≈ 0.026) — linear signal is basically absent
fig, ax = plt.subplots(figsize=(7, 5))
yield_corr.head(15).plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.invert_yaxis()
ax.set(title="Top 15 Features — |Pearson r| with Yield", xlabel="|r|")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_09_top_correlations.png", dpi=150)
plt.show()

## Feature Engineering

In [ ]:
# extract useful time features from planting/harvest dates
df["Planting_Date"] = pd.to_datetime(df["Planting_Date"], errors="coerce")
df["Harvest_Date"]  = pd.to_datetime(df["Harvest_Date"],  errors="coerce")

df["Crop_Duration"]  = (df["Harvest_Date"] - df["Planting_Date"]).dt.days
df["Planting_Month"] = df["Planting_Date"].dt.month
df["Harvest_Month"]  = df["Harvest_Date"].dt.month

df.drop(columns=["Planting_Date", "Harvest_Date"], inplace=True)

In [ ]:
# note: Crop_Duration is 91 for every row — dates are identical across the dataset
# Planting_Month and Harvest_Month also constant as a result
df[["Crop_Duration", "Planting_Month", "Harvest_Month"]].describe()

## Preprocessing

In [ ]:
X = df.drop(columns=["Yield"])
y = df["Yield"]

In [ ]:
num_features = X.select_dtypes(include="number").columns.tolist()
cat_features = X.select_dtypes(include="object").columns.tolist()

print(f"Numerical : {len(num_features)} features")
print(f"Categorical: {len(cat_features)} features → {cat_features}")

In [ ]:
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

In [ ]:
# sparse_output=False keeps the output as a dense array throughout
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

In [ ]:
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features),
])

In [ ]:
# 80/20 split, random_state=42 for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape}   Test: {X_test.shape}")

In [ ]:
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)
print(f"Processed — Train: {X_train_proc.shape},  Test: {X_test_proc.shape}")

In [ ]:
# build named dataframes for readability and export
feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_proc, columns=feature_names)
X_test_df  = pd.DataFrame(X_test_proc,  columns=feature_names)
X_train_df.head()

In [ ]:
# save for teammates working on Week 3-4 models
X_train_df.to_csv("../data/processed/X_train_processed.csv", index=False)
X_test_df.to_csv( "../data/processed/X_test_processed.csv",  index=False)
y_train.to_csv(   "../data/processed/y_train.csv",           index=False)
y_test.to_csv(    "../data/processed/y_test.csv",            index=False)
print("saved to data/processed/")

## Baseline Models

Running Linear Regression, Ridge, and Lasso as a quick sanity check before moving to ensemble methods.
Setting the performance floor so we know what to beat in Week 3.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge (α=1.0)":     Ridge(alpha=1.0),
    "Lasso (α=0.1)":     Lasso(alpha=0.1, max_iter=10000),
}

In [ ]:
results = []
fitted = {}   # store fitted models for diagnostic plots

for name, model in models.items():
    model.fit(X_train_proc, y_train)
    preds = model.predict(X_test_proc)
    fitted[name] = (model, preds)

    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    mae   = mean_absolute_error(y_test, preds)
    r2    = r2_score(y_test, preds)
    cv_r2 = cross_val_score(model, X_train_proc, y_train, cv=5, scoring="r2")

    results.append({
        "Model":  name,
        "RMSE":   round(rmse, 4),
        "MAE":    round(mae, 4),
        "R²":     round(r2, 4),
        "CV R²":  round(cv_r2.mean(), 4),
        "CV std": round(cv_r2.std(), 4),
    })

    print(f"{name:22s}  RMSE={rmse:.3f}  MAE={mae:.3f}  "
          f"R²={r2:.4f}  CV R²={cv_r2.mean():.4f} ± {cv_r2.std():.4f}")

In [ ]:
# R² ≈ 0 across the board — as expected given the weak linear correlations
# this sets the baseline; tree-based models should do much better in Week 3
results_df = pd.DataFrame(results).set_index("Model")
results_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
colors = ["steelblue", "coral", "mediumseagreen"]

for ax, metric, color in zip(axes, ["RMSE", "MAE", "R²"], colors):
    results_df[metric].plot(kind="bar", ax=ax, color=color,
                            edgecolor="white", rot=20)
    ax.set_title(metric)
    ax.set_xlabel("")

plt.suptitle("Baseline Model Comparison", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_10_baseline_comparison.png", dpi=150)
plt.show()

### Predicted vs Actual

If the model worked well, points should fall along the diagonal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, (_, preds)) in zip(axes, fitted.items()):
    ax.scatter(y_test, preds, alpha=0.3, s=10, color="steelblue", edgecolors="none")
    # perfect-prediction line
    lims = [y_test.min(), y_test.max()]
    ax.plot(lims, lims, "r--", linewidth=1.2, label="perfect fit")
    ax.set(title=name, xlabel="Actual Yield", ylabel="Predicted Yield")
    ax.legend(fontsize=8)

plt.suptitle("Predicted vs Actual — Baseline Models", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_11_predicted_vs_actual.png", dpi=150)
plt.show()

### Residual Plots

Residuals should be randomly scattered around 0 with no pattern. Any funnel shape or curve would indicate a problem with the model.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for col, (name, (_, preds)) in enumerate(fitted.items()):
    residuals = y_test.values - preds

    # residuals vs fitted
    axes[0, col].scatter(preds, residuals, alpha=0.3, s=10,
                         color="coral", edgecolors="none")
    axes[0, col].axhline(0, color="black", linewidth=1, linestyle="--")
    axes[0, col].set(title=f"{name}\nResiduals vs Fitted",
                     xlabel="Fitted", ylabel="Residual")

    # residual histogram
    axes[1, col].hist(residuals, bins=40, color="mediumpurple", edgecolor="white")
    axes[1, col].axvline(0, color="black", linewidth=1, linestyle="--")
    axes[1, col].set(title="Residual Distribution",
                     xlabel="Residual", ylabel="Count")

plt.suptitle("Residual Diagnostics — Baseline Models", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_12_residuals.png", dpi=150)
plt.show()